# Composites

## What you'll learn

- Define reusable operation compositions with `CompositeDefinition`
- Run a composite with `pipeline.run_composite()` — each internal operation becomes a real pipeline step
- Forward composite-level execution overrides as defaults to every child step
- Nest composites inside composites

**Prerequisites:** [Sources and Sequencing](01-sources-and-sequencing.ipynb), [Diamonds and Iteration](06-diamonds-and-iteration.ipynb)
**Estimated time:** 15 minutes

---

When operations are tightly coupled — for example, transform → score
where you always score immediately after transforming — defining them
as a **composite** lets you name that wiring once and reuse it anywhere.

A `CompositeDefinition` declares its inputs, outputs, and internal
wiring via a `compose()` method. Running it expands the wiring into real
pipeline steps: each `ctx.run()` becomes its own step with independent
caching, batching, worker dispatch, and provenance. Step names are
prefixed with the composite name so the grouping stays visible.

In [ ]:
from __future__ import annotations

from artisan.composites import (
    CompositeContext,
    CompositeDefinition,
)
from artisan.operations.examples import (
    DataGenerator,
    DataTransformer,
    MetricCalculator,
)
from artisan.orchestration import PipelineManager
from artisan.utils import tutorial_setup
from artisan.visualization import (
    build_macro_graph,
    build_micro_graph,
    inspect_pipeline,
)

> **Graph legend:** See [Sources and Sequencing](01-sources-and-sequencing.ipynb) for box/arrow key.

## Defining a composite

A composite is a subclass of `CompositeDefinition`. It declares:

- **`InputRole` / `OutputRole`** — StrEnum classes naming the composite's external inputs and outputs
- **`inputs` / `outputs`** — ClassVar dicts mapping roles to `InputSpec` / `OutputSpec`
- **`Params`** — an optional Pydantic `BaseModel` for composite-level parameters
- **`compose()`** — the wiring method that calls `ctx.run()` to execute internal operations

Here's a composite that transforms data and then computes quality metrics:

In [ ]:
from enum import StrEnum
from typing import ClassVar

from pydantic import BaseModel, Field

from artisan.schemas.specs.input_spec import InputSpec
from artisan.schemas.specs.output_spec import OutputSpec


class TransformAndScore(CompositeDefinition):
    """Transform data and compute quality metrics."""

    name = "transform_and_score"
    description = "Transform data, then compute metrics."

    class InputRole(StrEnum):
        DATASET = "dataset"

    class OutputRole(StrEnum):
        METRICS = "metrics"

    inputs: ClassVar[dict[str, InputSpec]] = {
        InputRole.DATASET: InputSpec(artifact_type="data", required=True),
    }
    outputs: ClassVar[dict[str, OutputSpec]] = {
        OutputRole.METRICS: OutputSpec(
            artifact_type="metric", description="Quality metrics"
        ),
    }

    class Params(BaseModel):
        scale_factor: float = Field(
            default=2.0, description="Scale factor for transform"
        )

    params: Params = Params()

    def compose(self, ctx: CompositeContext) -> None:
        transformed = ctx.run(
            DataTransformer,
            inputs={"dataset": ctx.input("dataset")},
            params={
                "scale_factor": self.params.scale_factor,
                "variants": 1,
                "seed": 100,
            },
        )
        scored = ctx.run(
            MetricCalculator,
            inputs={"dataset": transformed.output("dataset")},
        )
        ctx.output("metrics", scored.output("metrics"))

The composite is now a reusable building block. It accepts datasets,
runs two internal operations, and exposes only the final metrics.
The `compose()` method uses `CompositeContext` to wire everything
together — `ctx.input()` references declared inputs, `ctx.run()`
executes operations, and `ctx.output()` maps results to declared outputs.

## Baseline: hand-wired steps

First, build a three-step pipeline the traditional way — generate
data, transform it, then score it. Every pipeline that needs this
sequence re-types the same `pipeline.run()` calls and the wiring
between them.

```
generate ──→ transform ──→ score
   (3 datasets)  (3 datasets)  (3 metrics)
```

A composite lets you name this sequence once and reuse it.

In [ ]:
env_baseline = tutorial_setup("baseline")

In [ ]:
pipeline = PipelineManager.create(
    name="baseline",
    delta_root=env_baseline.delta_root,
    staging_root=env_baseline.staging_root,
    working_root=env_baseline.working_root,
)
output = pipeline.output

# Step 0: Generate 3 datasets
pipeline.run(operation=DataGenerator, name="generate", params={"count": 3, "seed": 42})

# Step 1: Transform each dataset
pipeline.run(
    operation=DataTransformer,
    name="transform",
    inputs={"dataset": output("generate", "datasets")},
    params={"scale_factor": 2.0, "variants": 1, "seed": 100},
)

# Step 2: Score each transformed dataset
pipeline.run(
    operation=MetricCalculator,
    name="score",
    inputs={"dataset": output("transform", "dataset")},
)

result_baseline = pipeline.finalize()

In [ ]:
inspect_pipeline(env_baseline.delta_root)

Three steps, and the wiring between them lives in this script. Any other
pipeline that needs generate → transform → score has to repeat it. A
composite captures the same wiring as one named, reusable unit.

In [ ]:
build_macro_graph(env_baseline.delta_root)

In [ ]:
build_micro_graph(env_baseline.delta_root)

## Running a composite: `pipeline.run_composite()`

`pipeline.run_composite()` expands the composite into real pipeline
steps — one per internal `ctx.run()`. Each step is prefixed with the
composite name, gets its own Delta Lake commit, and is cached and
dispatched independently.

```
generate ──→ transform_and_score.data_transformer ──→ transform_and_score.metric_calculator
   (3 datasets)             (3 datasets)                          (3 metrics)
```

The payoff is reuse: the same `TransformAndScore` definition drives this
expansion in every pipeline that uses it.

In [ ]:
env_composite = tutorial_setup("composite")

In [ ]:
pipeline = PipelineManager.create(
    name="composite",
    delta_root=env_composite.delta_root,
    staging_root=env_composite.staging_root,
    working_root=env_composite.working_root,
)
output = pipeline.output

# Step 0: Generate 3 datasets
pipeline.run(operation=DataGenerator, name="generate", params={"count": 3, "seed": 42})

# Run the composite — each internal operation becomes its own step
pipeline.run_composite(
    TransformAndScore,
    inputs={"dataset": output("generate", "datasets")},
    params={"scale_factor": 2.0},
)

result_composite = pipeline.finalize()

In [ ]:
inspect_pipeline(env_composite.delta_root)

In [ ]:
build_macro_graph(env_composite.delta_root)

In [ ]:
build_micro_graph(env_composite.delta_root)

The composite expanded into two named steps —
`transform_and_score.data_transformer` and
`transform_and_score.metric_calculator` — producing the same 3 metrics
as the baseline. Each internal operation got its own Delta Lake commit
and can be cached independently. The macro graph groups them under the
composite name, so the pipeline still reads as one logical unit.

## Forwarding execution overrides

`run_composite()` accepts the same execution overrides an ordinary step
takes — `step_runner`, `runner_resources`, `batch_strategy`,
`environment`, and so on. Each becomes the **default for every child
step**. A `ctx.run()` inside `compose()` that sets the same knob wins for
that step; anything it leaves unset falls back to the composite-level
default.

Here the composite-level `batch_strategy` sets the default batching for
both internal steps:

In [ ]:
env_forward = tutorial_setup("forward")

In [ ]:
pipeline = PipelineManager.create(
    name="forward",
    delta_root=env_forward.delta_root,
    staging_root=env_forward.staging_root,
    working_root=env_forward.working_root,
)
output = pipeline.output

pipeline.run(operation=DataGenerator, name="generate", params={"count": 4, "seed": 42})

# batch_strategy here is the default for every child step of the composite
pipeline.run_composite(
    TransformAndScore,
    inputs={"dataset": output("generate", "datasets")},
    params={"scale_factor": 2.0},
    batch_strategy={"artifacts_per_unit": 2},
)

result_forward = pipeline.finalize()

In [ ]:
inspect_pipeline(env_forward.delta_root)

Both child steps inherited the composite-level `batch_strategy` as their
default. If `TransformAndScore.compose()` set `batch_strategy` on a
specific `ctx.run()`, that value would win for that step only — the
composite-level value stays the default for every other step.

## Nesting composites

Composites can contain other composites. The inner composite's internal
operations expand into their own steps within the outer composite's
expansion, with dot-separated names. Here's a composite that generates
data and then applies `TransformAndScore`:

In [ ]:
class GenerateAndScore(CompositeDefinition):
    """Generate data, then transform and score it."""

    name = "generate_and_score"
    description = "Full pipeline: generate, transform, score."

    class OutputRole(StrEnum):
        METRICS = "metrics"

    outputs: ClassVar[dict[str, OutputSpec]] = {
        OutputRole.METRICS: OutputSpec(
            artifact_type="metric", description="Quality metrics"
        ),
    }

    class Params(BaseModel):
        count: int = Field(default=2, description="Number of datasets to generate")
        scale_factor: float = Field(
            default=2.0, description="Scale factor for transform"
        )

    params: Params = Params()

    def compose(self, ctx: CompositeContext) -> None:
        # Generate data
        generated = ctx.run(
            DataGenerator,
            params={"count": self.params.count, "seed": 42},
        )
        # Nest TransformAndScore composite
        scored = ctx.run(
            TransformAndScore,
            inputs={"dataset": generated.output("datasets")},
            params={"scale_factor": self.params.scale_factor},
        )
        ctx.output("metrics", scored.output("metrics"))

In [ ]:
env_nested = tutorial_setup("nested")

In [ ]:
pipeline = PipelineManager.create(
    name="nested",
    delta_root=env_nested.delta_root,
    staging_root=env_nested.staging_root,
    working_root=env_nested.working_root,
)

# Run the nested composite — every internal operation expands into a step
pipeline.run_composite(GenerateAndScore, params={"count": 2, "scale_factor": 1.5})

result_nested = pipeline.finalize()

In [ ]:
inspect_pipeline(env_nested.delta_root)

In [ ]:
build_macro_graph(env_nested.delta_root)

In [ ]:
build_micro_graph(env_nested.delta_root)

The outer composite (`GenerateAndScore`) ran `DataGenerator`, then
delegated to the inner composite (`TransformAndScore`), which ran
`DataTransformer` and `MetricCalculator`. Each internal operation became
its own pipeline step, named with the nested composite prefix. Nesting
lets you build larger composites from smaller ones while keeping each
piece independently testable.

## Summary

This tutorial covered composable operations with `CompositeDefinition`:

- **Composite definition** — subclass `CompositeDefinition`, declare
  `InputRole`/`OutputRole` enums, `inputs`/`outputs` specs, optional
  `Params`, and implement `compose()` using `CompositeContext`.
- **Running a composite** — `pipeline.run_composite(MyComposite, ...)`
  expands the composite into real pipeline steps, one per internal
  `ctx.run()`, each with its own caching, batching, dispatch, and
  provenance. Step names are prefixed with the composite name.
- **Forwarding overrides** — composite-level execution overrides
  (`step_runner`, `runner_resources`, `batch_strategy`, `environment`, …)
  become defaults for every child step; a per-op `ctx.run()` value wins
  for the knob it sets.
- **Nesting** — composites can contain other composites; the inner
  operations expand into their own steps with dot-separated names.

**Key takeaway:** Define your composition once as a `CompositeDefinition`,
then reuse it anywhere. Running it expands the wiring into ordinary
pipeline steps. Placement remains a runner concern: an optional provider
instance passed as `step_runner` is forwarded to every child step.

## Next steps

- [Composites and Composition](../../concepts/composites-and-composition.md) — conceptual deep dive into how composites execute and relate to operations
- [Writing Composite Operations](../../how-to-guides/writing-composite-operations.md) — step-by-step guide to building your own composites
- [CompositeDefinition Reference](../../reference/composite-definition.md) — API signatures and field tables
- [Error Handling in Practice](../05-errors-and-control/02-error-visibility.ipynb) — runtime failures, failure logs, and FailurePolicy
- [Diamonds and Iteration](06-diamonds-and-iteration.ipynb) — diamond DAGs and iterative refinement loops